# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [1]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/bin/python
Financial_Fraud_Detection_Thesis


In [2]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 12
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = True       # dual-tower (change B)

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


PyTorch 2.11.0  device=mps


In [3]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

mps:0


## 5. Build per-UID windows on the combined frame

In [ ]:
data = np.load("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz")

X_train_seq = data["X_train_seq"]
L_train = data["L_train"]
X_test_seq = data["X_test_seq"]
L_test = data["L_test"]
train_order = data["train_order"]
test_order = data["test_order"]
train_pos = data["train_pos"]
test_pos = data["test_pos"]
y_aligned = data["y_aligned"]
dt_m_aligned = data["dt_m_aligned"]

## 6. PyTorch `Dataset` and `DataLoader`

A thin wrapper around the numpy arrays. We hand the Dataset whatever subset of indices
the current fold needs, so we don't have to copy big arrays.


In [80]:
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = X                                            # keep as mmap np.ndarray
        self.lengths = torch.from_numpy(lengths.astype('int64'))
        self.y = None if y is None else torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i], self.lengths[i]
        return self.X[i], self.lengths[i], self.y[i]

In [81]:
def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(
        WindowDataset(X, lengths, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

## 7. PyTorch LSTM model

This is the structural mirror of arunmohan003's `SentimentRNN` — same `__init__` /
`forward` / `init_hidden` pattern — adapted for numeric input (no `nn.Embedding`).


In [82]:
class FraudLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 drop_prob=DROPOUT, output_dim=1,
                 use_static_tower=USE_STATIC_TOWER,
                 bidirectional=True):                     # NEW
        super().__init__()
        self.use_static_tower = use_static_tower
        self.bidirectional   = bidirectional             # NEW

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,                 # NEW
        )

        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)   # NEW
        seq_out_dim  = 2 * lstm_out_dim                            # mean + max

        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64),         nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths):
        # x: (B, T, F)  left-padded; lengths: (B,) real-step counts
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)                                  # (B, T, H)

        # build a mask for the real (right-aligned) positions
        idx = torch.arange(T, device=x.device).unsqueeze(0)         # (1, T)
        pos_from_end = T - 1 - idx                                  # 0..T-1
        mask = pos_from_end < lengths.to(x.device).unsqueeze(1)     # (B, T)
        mask_f = mask.unsqueeze(-1).float()

        # masked mean
        sum_  = (lstm_out * mask_f).sum(dim=1)
        cnt   = mask_f.sum(dim=1).clamp(min=1.0)
        mean_pool = sum_ / cnt

        # masked max (pads → very negative)
        neg_inf  = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        seq_vec = torch.cat([mean_pool, max_pool], dim=1)

        if self.use_static_tower:
            current  = x[:, -1, :]                                  # last real step
            stat_vec = self.static_mlp(current)
            feat = torch.cat([seq_vec, stat_vec], dim=1)
        else:
            feat = seq_vec

        return self.head(feat).squeeze(-1)

In [83]:
# smoke test
N_FEATURES = X_train_seq.shape[2]
m = FraudLSTM(N_FEATURES).to(device)
xb = torch.randn(8, WINDOW, N_FEATURES, device=device)
lb = torch.randint(1, WINDOW+1, (8,), device=device)
print('forward smoke test out shape:', m(xb, lb).shape)
del m, xb, lb

forward smoke test out shape: torch.Size([8])


## 8. Expanding-window CV + training loop

Same fold scheme as the Keras notebook. Training loop mirrors arunmohan003's pattern:

1. Per epoch, iterate batches and reset hidden state for each batch (windows are
   independent).
2. `optimizer.zero_grad()` → `forward` → `loss.backward()` → `clip_grad_norm_` →
   `optimizer.step()`.
3. Validation pass with `model.eval()` and `torch.no_grad()`.
4. Early-stop on validation AUC, restore best weights.


In [84]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)


def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)

device: mps


In [85]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTM(n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [86]:
# ===== run folds with seed ensembling =====
oof        = np.full(len(X_train_seq), np.nan, dtype=np.float32)
test_preds = np.zeros(len(X_test_seq), dtype=np.float32)
fold_aucs  = []

fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f'{len(fold_specs)} folds; {N_SEEDS} seeds per fold')

for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
    print(f'\n=== Fold {fold}: train {tm} → validate {vm} '
          f'(train={len(idxT):,}, valid={len(idxV):,}) ===')

    seed_val_preds, seed_test_preds = [], []
    for s in range(N_SEEDS):
        seed = SEED + s
        print(f'-- seed {seed} --')
        vp, va, model = train_one_fold(
            X_train_seq[idxT], L_train[idxT], y_aligned[idxT],
            X_train_seq[idxV], L_train[idxV], y_aligned[idxV],
            n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
            lr=LR, weight_decay=WEIGHT_DECAY, device=device,
            early_stop_patience=EARLY_STOP_PATIENCE,
            grad_clip=GRAD_CLIP, seed=seed,
        )
        seed_val_preds.append(vp)

        # test preds from this seed's best model
        model.eval()
        test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
        tps = []
        with torch.no_grad():
            for xb, lb in test_loader:
                xb = xb.to(device); lb = lb.to(device)
                tps.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        seed_test_preds.append(np.concatenate(tps))

        del model
        if device.type == 'mps': torch.mps.empty_cache()
        gc.collect()

    fold_val   = np.mean(seed_val_preds, axis=0)
    fold_test  = np.mean(seed_test_preds, axis=0)
    fold_auc   = roc_auc_score(y_aligned[idxV], fold_val)
    print(f'   fold AUC (seed-avg) = {fold_auc:.4f}')
    fold_aucs.append((int(vm), float(fold_auc)))
    oof[idxV]  = fold_val
    test_preds += fold_test

if len(fold_specs):
    test_preds /= len(fold_specs)

validated = ~np.isnan(oof)
overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
print(f'\n=== LSTM OOF AUC (validated months only) = {overall_auc:.4f} ===')
print(f'   per-fold: {fold_aucs}')

3 folds; 3 seeds per fold

=== Fold 0: train [12, 13, 14] → validate 15 (train=315,927, valid=101,632) ===
-- seed 42 --
   ep  1/30  loss=0.2629  val_auc=0.8517  (51.4s)
   ep  2/30  loss=0.0933  val_auc=0.8755  (47.8s)
   ep  3/30  loss=0.0801  val_auc=0.8802  (46.8s)
   ep  4/30  loss=0.0719  val_auc=0.8795  (49.1s)
   ep  5/30  loss=0.0652  val_auc=0.8742  (45.8s)
   ep  6/30  loss=0.0598  val_auc=0.8875  (44.3s)
   ep  7/30  loss=0.0556  val_auc=0.8551  (48.5s)
   ep  8/30  loss=0.0524  val_auc=0.8679  (47.7s)
   ep  9/30  loss=0.0490  val_auc=0.8652  (50.5s)
   ep 10/30  loss=0.0464  val_auc=0.8639  (50.2s)
   ep 11/30  loss=0.0436  val_auc=0.8589  (46.1s)
   ep 12/30  loss=0.0411  val_auc=0.8523  (46.7s)
   ep 13/30  loss=0.0389  val_auc=0.8517  (47.5s)
   ep 14/30  loss=0.0362  val_auc=0.8439  (49.9s)
   ep 15/30  loss=0.0351  val_auc=0.8447  (51.4s)
   ep 16/30  loss=0.0322  val_auc=0.8300  (47.2s)
   ep 17/30  loss=0.0301  val_auc=0.8362  (46.9s)
   ep 18/30  loss=0.0286  val